# Airline Passenger Satisfaction | Modeling & Kaggle Submissions

**Kaggle Playground Series — Predicting Airline Satisfaction**  
**Task:** Binary classification · **Evaluation:** ROC-AUC · **Models:** Logistic Regression, Histogram Gradient Boosting, CatBoost

## Objective
Predict the probability that a passenger is satisfied (`satisfaction = 1`) using airline-service ratings and passenger/travel characteristics. Compare increasingly capable models using the **same stratified validation split**, then generate reproducible Kaggle submissions.

> This notebook builds on the exploratory analysis in `01_eda.ipynb`. Competition test labels are not available locally, so all model comparisons use a held-out portion of `train.csv`.

### Results at a glance

| Experiment | Model | Validation ROC-AUC | Kaggle public ROC-AUC |
|:--|:--|--:|--:|
| 1 | Logistic Regression | 0.92046 | 0.92229 |
| 2 | Histogram Gradient Boosting | 0.95652 | 0.95696 |
| 3 | CatBoost | **0.95730** | **0.95777** |

*The public leaderboard scores above were recorded from competition submissions; they are not generated by running this notebook.*

## 1. Setup and data loading

The notebook assumes it is executed with `notebooks/` as the working directory, using `../data/raw/` for downloaded Kaggle files and `../submissions/` for generated CSVs.

**Dependencies:** `pandas`, `numpy`, `scikit-learn` (with `OneHotEncoder(sparse_output=...)` support), and `catboost`. Install from your project environment rather than inside this notebook.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from catboost import CatBoostClassifier

DATA_DIR = Path("../data/raw")
SUBMISSIONS_DIR = Path("../submissions")

In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
submission_template = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Training shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

Training shape: (699635, 23)
Test shape: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


## 2. Features and validation design

Drop the unique passenger `id` from predictors and keep `satisfaction` as the binary target. The four passenger/travel descriptors are categorical; all other predictors are treated as numerical (including ordinal service ratings).

An **80/20 stratified split** keeps class proportions similar in both subsets. Crucially, preprocessing is fitted *inside* model pipelines on the training subset only, avoiding preprocessing leakage.

In [7]:
X = train_df.drop(columns=["id", "satisfaction"])
y = train_df["satisfaction"].astype(int)

categorical_features = ["Gender", "Customer Type", "Type of Travel", "Class"]
numerical_features = X.select_dtypes(include="number").columns.tolist()

assert set(categorical_features + numerical_features) == set(X.columns)
assert not (set(categorical_features) & set(numerical_features))
assert set(y.unique()).issubset({0, 1})

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_test = test_df.drop(columns="id")
assert list(X_test.columns) == list(X.columns), "Train/test feature columns differ"
print(f"Train: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

Train: (559708, 21) | Validation: (139927, 21) | Test: (299844, 21)


## 3. Experiment 1 — Logistic Regression baseline

A regularized linear classifier establishes a simple benchmark.

- **Numerical:** median imputation, followed by standardization.
- **Categorical:** most-frequent imputation, followed by one-hot encoding (`handle_unknown='ignore'`).
- **Estimator:** Logistic Regression with an increased iteration cap for convergence.

The pipeline fits all transformations only on `X_train`; `predict_proba` produces the positive-class probabilities used for ROC-AUC.

In [9]:
lr_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
lr_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
lr_preprocessor = ColumnTransformer([
    ("numerical", lr_numeric, numerical_features),
    ("categorical", lr_categorical, categorical_features),
])
model_pipeline = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

In [10]:
model_pipeline.fit(X_train, y_train)
y_val_proba = model_pipeline.predict_proba(X_val)[:, 1]
validation_auc = roc_auc_score(y_val, y_val_proba)
print(f"Logistic Regression validation ROC-AUC: {validation_auc:.5f}")

Logistic Regression validation ROC-AUC: 0.92046


**Baseline interpretation.** A validation ROC-AUC of approximately **0.92046** shows strong ability to rank satisfied passengers above dissatisfied passengers. It does *not* mean that 92% of individual predictions are correct. The corresponding first Kaggle public score was **0.92229**.

## 4. Experiment 2 — Histogram Gradient Boosting

Boosted decision trees can capture **nonlinear patterns and interactions** between variables without manually specifying those interactions.

For this experiment, the categorical encoder produces dense output (`sparse_output=False`), compatible with `HistGradientBoostingClassifier`. Numerical scaling is retained for continuity with the baseline, although trees do not require it. This experiment uses a **separate preprocessor object** so changes cannot accidentally affect the baseline definition.

In [13]:
hgb_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
hgb_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
hgb_preprocessor = ColumnTransformer([
    ("numerical", hgb_numeric, numerical_features),
    ("categorical", hgb_categorical, categorical_features),
])
hgb_pipeline = Pipeline([
    ("preprocessor", hgb_preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=42)),
])

In [14]:
hgb_pipeline.fit(X_train, y_train)
y_val_proba_hgb = hgb_pipeline.predict_proba(X_val)[:, 1]
hgb_auc = roc_auc_score(y_val, y_val_proba_hgb)
print(f"HGB validation ROC-AUC: {hgb_auc:.5f}")

HGB validation ROC-AUC: 0.95652


**Result.** Histogram Gradient Boosting improved validation ROC-AUC to **0.95652** (from 0.92046), suggesting that nonlinear patterns or feature interactions are important. Its recorded Kaggle public score was **0.95696**. This is an association between model choice and performance, not direct evidence of which individual interactions matter.

## 5. Experiment 3 — CatBoost

CatBoost is another gradient-boosting algorithm designed to work effectively with categorical columns. Unlike the first two experiments, **no one-hot encoding or scaling is applied**: we pass the original feature DataFrame and identify its categorical columns via `cat_features`.

CatBoost can handle missing numerical values, but actual `NaN` values in categorical columns need explicit treatment. We check for those before fitting. The model monitors validation ROC-AUC and stops if there is no improvement for 50 rounds.

In [17]:
missing_cats = X_train[categorical_features].isna().sum()
missing_cats_val = X_val[categorical_features].isna().sum()
missing_cats_test = X_test[categorical_features].isna().sum()
print(pd.DataFrame({
    "train_missing": missing_cats,
    "val_missing": missing_cats_val,
    "test_missing": missing_cats_test,
}))
assert not missing_cats.any() and not missing_cats_val.any() and not missing_cats_test.any(), (
    "Missing categorical values found: fill consistently before using CatBoost."
)

                train_missing  val_missing  test_missing
Gender                      0            0             0
Customer Type               0            0             0
Type of Travel              0            0             0
Class                       0            0             0


In [18]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    cat_features=categorical_features,
)

In [19]:
catboost_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
)

0:	test: 0.9326591	best: 0.9326591 (0)	total: 357ms	remaining: 5m 56s
100:	test: 0.9536938	best: 0.9536938 (100)	total: 19.1s	remaining: 2m 50s
200:	test: 0.9553362	best: 0.9553362 (200)	total: 38.6s	remaining: 2m 33s
300:	test: 0.9560687	best: 0.9560687 (300)	total: 58.4s	remaining: 2m 15s
400:	test: 0.9565196	best: 0.9565196 (400)	total: 1m 17s	remaining: 1m 56s
500:	test: 0.9567251	best: 0.9567251 (500)	total: 1m 36s	remaining: 1m 36s
600:	test: 0.9569421	best: 0.9569421 (600)	total: 1m 56s	remaining: 1m 17s
700:	test: 0.9570884	best: 0.9570884 (700)	total: 2m 15s	remaining: 57.9s
800:	test: 0.9571890	best: 0.9571900 (799)	total: 2m 35s	remaining: 38.7s
900:	test: 0.9572463	best: 0.9572469 (899)	total: 2m 55s	remaining: 19.3s
999:	test: 0.9572988	best: 0.9573018 (979)	total: 3m 14s	remaining: 0us

bestTest = 0.9573017502
bestIteration = 979

Shrink model to first 980 iterations.


In [20]:
y_val_proba_cat = catboost_model.predict_proba(X_val)[:, 1]
catboost_auc = roc_auc_score(y_val, y_val_proba_cat)
print(f"CatBoost validation ROC-AUC: {catboost_auc:.5f}")
print(f"Best iteration: {catboost_model.best_iteration_}")

CatBoost validation ROC-AUC: 0.95730
Best iteration: 979


**Result.** CatBoost achieved validation ROC-AUC **0.95730**, with best iteration **979** (original run). Its recorded Kaggle public score was **0.95777**. The advantage over HGB is only about **0.00078 validation ROC-AUC**, so a single split is insufficient to establish a robust superiority claim. Also, validation data informed CatBoost's early stopping, making that validation score partly model-selection-dependent.

## 6. Model comparison

All three models share the same train-validation partition. The notebook prints scores **from the current execution**, rather than displaying only manually entered results.

In [23]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "HistGradientBoosting", "CatBoost"],
    "Validation ROC-AUC": [validation_auc, hgb_auc, catboost_auc],
    "Kaggle public ROC-AUC (recorded)": [0.92229, 0.95696, 0.95777],
})
results.sort_values("Validation ROC-AUC", ascending=False).reset_index(drop=True)

,Model,Validation ROC-AUC,Kaggle public ROC-AUC (recorded)
0,CatBoost,0.957302,0.95777
1,HistGradientBoosting,0.956520,0.95696
2,Logistic Regression,0.920455,0.92229


## 7. Create and validate Kaggle submissions

Generate probabilities from each **already-fitted** model. A single helper checks the submission schema, ID order, row count, missing values, and probability bounds **before writing anything**. All three outputs use separate filenames so earlier experiments remain reproducible.

The original Kaggle `sample_submission.csv` is read-only here: never write results into `data/raw/`.

In [25]:
def save_submission(probabilities, filename):
    probabilities = np.asarray(probabilities)
    submission = submission_template.copy()

    assert list(submission.columns) == ["id", "satisfaction"], "Unexpected submission schema"
    assert len(probabilities) == len(test_df) == len(submission), "Prediction count mismatch"
    assert submission["id"].reset_index(drop=True).equals(
        test_df["id"].reset_index(drop=True)
    ), "Test and submission IDs differ or are out of order"
    assert np.isfinite(probabilities).all(), "Predictions contain NaN or infinity"
    assert ((probabilities >= 0) & (probabilities <= 1)).all(), "Probabilities outside [0, 1]"

    submission["satisfaction"] = probabilities
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
    output_path = SUBMISSIONS_DIR / filename
    submission.to_csv(output_path, index=False)
    print(f"Validated and saved: {output_path} ({len(submission):,} rows)")
    return output_path

In [26]:
lr_submission_path = save_submission(
    model_pipeline.predict_proba(X_test)[:, 1], "submission_v1.csv"
)
hgb_submission_path = save_submission(
    hgb_pipeline.predict_proba(X_test)[:, 1], "submission_v2_hgb.csv"
)
cat_submission_path = save_submission(
    catboost_model.predict_proba(X_test)[:, 1], "submission_v3_catboost.csv"
)

Validated and saved: ..\submissions\submission_v1.csv (299,844 rows)
Validated and saved: ..\submissions\submission_v2_hgb.csv (299,844 rows)
Validated and saved: ..\submissions\submission_v3_catboost.csv (299,844 rows)


## 8. Findings and limitations

**Key findings**

- All three models performed well on the held-out set, with **CatBoost obtaining the highest measured ROC-AUC**.
- The largest jump came from Logistic Regression to tree-based boosting, consistent with the benefit of modeling nonlinearities and interactions.
- CatBoost's incremental gain over Histogram Gradient Boosting is **small**; it should not be overstated.
- Local validation scores were close to Kaggle's *public* scores on the recorded submissions, but that does not guarantee performance on the private leaderboard.

**Limitations**

- Only one train-validation split was used; no repeated or cross-validation estimate of variability was obtained.
- CatBoost used the validation set for early stopping, so it was not completely untouched during model selection.
- No dedicated feature-importance, group-wise error analysis, calibration study, or systematic hyperparameter search was performed.
- Public leaderboard scores are not a substitute for an independent final test set.

**Next steps (out of scope here):** investigate feature importance, segment-level performance, and a separate evaluation strategy if this model were to be deployed. The competition was treated as a focused end-to-end learning project rather than an exhaustive leaderboard optimization exercise.